# 02 Mordred Descriptor Extraction

Phase 2 calculates raw 2D Mordred descriptors for the Phase 1 Training and Test splits.

Run this notebook in the `mordred` conda environment. This is an extraction-only notebook: it does not remove missing values, zero-variance descriptors, correlated descriptors, rows, or columns. All filtering and feature selection are reserved for Phase 3.

In [1]:
from pathlib import Path

import pandas as pd
from rdkit import Chem
from mordred import Calculator, descriptors

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test.csv"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
TRAIN_OUT = FEATURE_DIR / "mordred_train.csv"
TEST_OUT = FEATURE_DIR / "mordred_test.csv"

assert TRAIN_PATH.exists(), f"Missing input file: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Missing input file: {TEST_PATH}"


## Load Phase 1 Splits

The Phase 1 split files are read as fixed inputs. The expected row counts are 514 Training compounds and 128 Test compounds.

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

expected_columns = ["SMILES", "logKoc"]
assert list(train_df.columns) == expected_columns, f"Unexpected train columns: {list(train_df.columns)}"
assert list(test_df.columns) == expected_columns, f"Unexpected test columns: {list(test_df.columns)}"
assert train_df.shape == (514, 2), f"Unexpected train shape: {train_df.shape}"
assert test_df.shape == (128, 2), f"Unexpected test shape: {test_df.shape}"

print(f"Train input shape: {train_df.shape}")
print(f"Test input shape: {test_df.shape}")
display(train_df.head())
display(test_df.head())


Train input shape: (514, 2)
Test input shape: (128, 2)


,SMILES,logKoc
0,C=CC=O,-0.31
1,CC(=O)O,0.00
2,CNC(=O)/C=C(\C)OP(=O)(OC)OC,0.00
3,CCO,0.20
4,C[C@H](O)CO,0.36


,SMILES,logKoc
0,C1CO1,0.34
1,O=c1ccc(=O)[nH][nH]1,0.45
2,C=O,0.56
3,CNC(=O)O/N=C/C(C)(C)S(C)(=O)=O,0.71
4,CN=C=S,0.97


## Convert SMILES to RDKit Molecules

The Phase 1 SMILES were already cleaned and canonicalized. This cell verifies that every SMILES still converts to an RDKit molecule before descriptor calculation.

In [3]:
def smiles_to_mols(df: pd.DataFrame, label: str) -> list:
    mols = [Chem.MolFromSmiles(smiles) for smiles in df["SMILES"]]
    invalid_indices = [idx for idx, mol in enumerate(mols) if mol is None]
    print(f"{label} invalid molecule count: {len(invalid_indices)}")
    if invalid_indices:
        display(df.iloc[invalid_indices][["SMILES", "logKoc"]])
    assert not invalid_indices, f"{label} contains invalid SMILES; aborting extraction."
    return mols


train_mols = smiles_to_mols(train_df, "Train")
test_mols = smiles_to_mols(test_df, "Test")

print(f"Train molecule count: {len(train_mols)}")
print(f"Test molecule count: {len(test_mols)}")


Train invalid molecule count: 0
Test invalid molecule count: 0
Train molecule count: 514
Test molecule count: 128


## Calculate Raw 2D Mordred Descriptors

Mordred is initialized as `Calculator(descriptors, ignore_3D=True)` to avoid 3D conformer generation. Train and Test descriptors are calculated separately. Mordred calculation errors or missing descriptor values are serialized as `NaN` through numeric coercion, but no rows or descriptor columns are removed.

In [4]:
calc = Calculator(descriptors, ignore_3D=True)
print(f"Mordred 2D descriptor count: {len(calc.descriptors)}")

print(f"Calculating Train descriptors for {len(train_mols)} molecules...")
train_desc = calc.pandas(train_mols)

print(f"Calculating Test descriptors for {len(test_mols)} molecules...")
test_desc = calc.pandas(test_mols)

# Keep every descriptor column. Coercion converts Mordred Missing/Error objects
# to NaN for CSV storage; it does not filter rows or features.
train_desc = train_desc.apply(pd.to_numeric, errors="coerce")
test_desc = test_desc.apply(pd.to_numeric, errors="coerce")

assert train_desc.shape[0] == len(train_df), "Train descriptor row count changed"
assert test_desc.shape[0] == len(test_df), "Test descriptor row count changed"
assert list(train_desc.columns) == list(test_desc.columns), "Train/Test descriptor schemas differ"

print(f"Raw Train descriptor shape: {train_desc.shape}")
print(f"Raw Test descriptor shape: {test_desc.shape}")
print(f"Train NaN values retained: {int(train_desc.isna().sum().sum())}")
print(f"Test NaN values retained: {int(test_desc.isna().sum().sum())}")
display(train_desc.head())


Mordred 2D descriptor count: 1613
Calculating Train descriptors for 514 molecules...


  0%|          | 0/514 [00:00<?, ?it/s]

  1%|          | 3/514 [00:00<00:18, 27.66it/s]

  4%|▎         | 19/514 [00:00<00:05, 90.71it/s]

  6%|▌         | 32/514 [00:00<00:04, 103.89it/s]

...

 88%|████████▊ | 113/128 [00:02<00:00, 24.08it/s]

 99%|█████████▉| 127/128 [00:05<00:00,  8.50it/s]

100%|██████████| 128/128 [00:05<00:00, 22.98it/s]

Raw Train descriptor shape: (514, 1613)
Raw Test descriptor shape: (128, 1613)
Train NaN values retained: 107639
Test NaN values retained: 27464


,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,2.121320,2.340100,0,0,4.472136,1.618034,3.236068,4.472136,1.118034,2.155909,...,5.509388,22.328143,56.026215,7.003277,10,1,10.0,8.0,2.500000,1.250000
1,2.449490,2.449490,1,0,3.464102,1.732051,3.464102,3.464102,0.866025,2.178059,...,6.188264,24.179697,60.021129,7.502641,9,0,12.0,9.0,3.111111,1.000000
2,9.570086,9.944926,0,0,16.030562,2.262124,4.524249,16.030562,1.145040,3.482048,...,9.082166,44.461407,223.060959,7.966463,332,17,60.0,64.0,7.534722,3.458333
3,1.414214,1.414214,0,0,2.828427,1.414214,2.828427,2.828427,0.942809,1.849457,...,4.174387,17.310771,46.041865,5.115763,4,0,6.0,4.0,2.250000,1.000000
4,3.047207,3.305183,0,0,5.226252,1.847759,3.695518,5.226252,1.045250,2.408576,...,6.834109,27.254130,76.052429,5.850187,18,2,16.0,14.0,3.361111,1.333333


## Merge Metadata and Save Raw Feature Tables

The output tables contain the original `SMILES` and `logKoc` columns followed by the complete raw Mordred descriptor matrix.

In [5]:
train_features = pd.concat(
    [train_df[["SMILES", "logKoc"]].reset_index(drop=True), train_desc.reset_index(drop=True)],
    axis=1,
)
test_features = pd.concat(
    [test_df[["SMILES", "logKoc"]].reset_index(drop=True), test_desc.reset_index(drop=True)],
    axis=1,
)

assert train_features.shape[0] == 514, f"Unexpected train output rows: {train_features.shape[0]}"
assert test_features.shape[0] == 128, f"Unexpected test output rows: {test_features.shape[0]}"
assert train_features.shape[1] == test_features.shape[1], "Train/Test output column counts differ"
assert list(train_features.columns) == list(test_features.columns), "Train/Test output columns differ"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
train_features.to_csv(TRAIN_OUT, index=False)
test_features.to_csv(TEST_OUT, index=False)

print(f"Saved Train Mordred descriptors: {TRAIN_OUT}")
print(f"Saved Test Mordred descriptors: {TEST_OUT}")
print(f"Train output shape: {train_features.shape}")
print(f"Test output shape: {test_features.shape}")


Saved Train Mordred descriptors: /home/jun/Documents/qsar_modeling/data/features/mordred_train.csv
Saved Test Mordred descriptors: /home/jun/Documents/qsar_modeling/data/features/mordred_test.csv
Train output shape: (514, 1615)
Test output shape: (128, 1615)
